# Real-data examples: old vs new trended-feature denominator

Purpose of this notebook: pull a chunk of real tradelines per bureau and show,
on concrete examples, how the effective month range (the denominator of every
`percent_<rate>_<window>_months` feature) differs between the shipping code and
the proposed fix.

**Note on the input files:** the parquet chunks under
`payment_processing_research_data/<bureau>/test/mapped/` already had the
asset's `mapping` block applied -- MapperV2 ran the StringConverter /
DateConverter / NumericConverter steps from each bureau's FE2 trade asset when
`map_and_save_mapped_data.ipynb` wrote them. So `rptDate`, `date_of_request`,
and the payment-pattern columns are already in their converted form here, and
we do NOT re-run MapperV2: `prep_bureau` below only adds the DateDiff column
and combines the pattern columns into `zest_payment_pattern`.

Denominator definitions being compared (same math as
`analyze_difference_in_denominator.ipynb`):

    eff_old = month_range          - count('#')
    eff_new = len(trimmed_pattern) - count('#') - count(missing_data_char)


In [1]:
import re
from pathlib import Path

import pandas as pd
import numpy as np

from feature_engine_parts.fe_parts_V2.preprocessors.payment_pattern_aggregator import PaymentPatternsAggregatorV2
from feature_engine_parts.fe_parts_V2.preprocessors.date_diff import DateDiffV2

from configs import EQUIFAX, EXPERIAN, TRANSUNION, mapped_dir
from helpers import (
    MISSING_DATA_CHARS,
    PROJECT_COLS,
    MONTH_RANGES,
    load_asset_json,
    get_aggregator_params,
)

SPLIT = 'test'

BUREAU_CFGS = {'equifax': EQUIFAX, 'experian': EXPERIAN, 'transunion': TRANSUNION}

In [3]:
def prep_bureau(bureau, trade_df_mapped):
    """Returns to_use_for_payment_processing with `zest_payment_pattern` populated.

    Input must already be MAPPED (the asset's `mapping` block applied) -- the
    mapped/ chunks are, so there is no MapperV2 step here.
    """
    asset = load_asset_json(bureau)
    cols = [c for c in PROJECT_COLS[bureau] if c in trade_df_mapped.columns]
    to_use_for_payment_processing = trade_df_mapped[cols].copy()

    # DateDiffV2: (date_of_request - rptDate) / 30.436875 days -> months_since_rptDate
    date_diff = DateDiffV2(feature='rptDate', reference_feature='date_of_request',
                           new_feature='months_since_rptDate')
    to_use_for_payment_processing = date_diff.transform(to_use_for_payment_processing)

    # Combine into zest_payment_pattern via the aggregator helper.
    agg = PaymentPatternsAggregatorV2(**get_aggregator_params(asset))
    to_use_for_payment_processing['zest_payment_pattern'] = agg._construct_payment_pattern_cols(
        to_use_for_payment_processing
    )
    return to_use_for_payment_processing

In [4]:
# Load ONE mapped chunk per bureau -- plenty of rows for examples.
# Paths come from configs.mapped_dir, not hardcoded.
mapped = {}
for bureau, cfg in BUREAU_CFGS.items():
    path = Path(mapped_dir(cfg, SPLIT)) / 'part-00000.parquet'
    mapped[bureau] = pd.read_parquet(path)
    print(f'[{bureau}]  {len(mapped[bureau]):,} rows from {path}')

[equifax]  100,000 rows from /home/jag/payment-processor-research/payment_processing_research_data/equifax/test/mapped/part-00000.parquet
[experian]  100,000 rows from /home/jag/payment-processor-research/payment_processing_research_data/experian/test/mapped/part-00000.parquet
[transunion]  100,000 rows from /home/jag/payment-processor-research/payment_processing_research_data/transunion/test/mapped/part-00000.parquet


In [5]:
# Transform all three bureaus: PROJECT_COLS -> DateDiff -> zest_payment_pattern.
prepped = {}
for bureau, df in mapped.items():
    prepped[bureau] = prep_bureau(bureau, df)
    sample = prepped[bureau]['zest_payment_pattern'].dropna()
    print(f'[{bureau}]  {len(prepped[bureau]):,} rows; '
          f'example pattern: {sample.iloc[0][:48] if len(sample) else "(none)"}')

/home/jag/.conda/envs/model_engine_2_py310/lib/python3.10/site-packages/zaml/common/utils/io.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


[equifax]  100,000 rows; example pattern: #11111111111111*********************************
[experian]  100,000 rows; example pattern: ############B00000000000000000000000000000000000
[transunion]  100,000 rows; example pattern: #11111111111111111111111111111111111111111111111


## Examples where the denominator changes

For each bureau: compute `eff_old` / `eff_new` for one illustrative window and
show a handful of real tradelines where they disagree, alongside the
`percent_DQ30+` value each method would produce. These are tradelines with
missing-data codes inside the window (or a pattern string shorter than the
window), i.e. exactly the cases the fix targets.

In [7]:
import numpy as np

In [8]:
SHOW_M     = 24   # window to illustrate
N_EXAMPLES = 8    # rows to show per bureau

examples = {}
for bureau, df in prepped.items():
  missing_char = MISSING_DATA_CHARS[bureau]
  dq30_codes   = get_aggregator_params(load_asset_json(bureau))['payment_patterns']['rate']['DQ30+']

  ppt     = df['zest_payment_pattern'].fillna('')
  trimmed = ppt.str[:SHOW_M]

  # OLD: nominal window minus '#' fillers.
  # NEW: observed string length minus '#' and the bureau's missing-data char.
  eff_old = SHOW_M             - trimmed.str.count('#')
  eff_new = trimmed.str.len()  - trimmed.str.count('#') - trimmed.str.count(re.escape(missing_char))

  n_dq30 = trimmed.str.count('|'.join(re.escape(c) for c in dq30_codes))

  out = pd.DataFrame({
      'ZEST_KEY':              df['ZEST_KEY'],
      'zest_payment_pattern':  ppt,        # original, before trimming to the window
      f'trimmed_{SHOW_M}':     trimmed,
      'eff_old':               eff_old,
      'eff_new':               eff_new,
      'n_DQ30+':               n_dq30,
      'pct_DQ30+_old':         (n_dq30 / eff_old).round(4),
      'pct_DQ30+_new':         (n_dq30 / eff_new.where(eff_new > 0)).round(4),
  })
  examples[bureau] = out
  out['diff'] = np.abs(out['pct_DQ30+_old']-out['pct_DQ30+_new'])
    
  changed = out[out['eff_old'] != out['eff_new']].sort_values(by= 'diff', ascending = False)
  print(f"\n[{bureau}]  missing_char={missing_char!r}  window={SHOW_M}m  "
        f"{len(changed):,}/{len(out):,} rows change "
        f"({len(changed) / len(out) * 100:.1f}%)")
  # most interesting examples first: a DQ30+ in the window AND a changed denominator
  show = changed.head(N_EXAMPLES)
  with pd.option_context('display.max_colwidth', None):
      display(show)


[equifax]  missing_char='*'  window=24m  31,731/100,000 rows change (31.7%)


,ZEST_KEY,zest_payment_pattern,trimmed_24,eff_old,eff_new,n_DQ30+,pct_DQ30+_old,pct_DQ30+_new,diff
28437,00001553210_27,#6***********************************************,#6**********************,23,1,1,0.0435,1.0,0.9565
49728,00001939053_13,#6***********************************************,#6**********************,23,1,1,0.0435,1.0,0.9565
55783,00009969428_15,#6***********************************************,#6**********************,23,1,1,0.0435,1.0,0.9565
88962,00002348248_19,#6***********************************************,#6**********************,23,1,1,0.0435,1.0,0.9565
36673,00025056043_2,#6***********************************************,#6**********************,23,1,1,0.0435,1.0,0.9565
79043,00029052239_14,#6***********************************************,#6**********************,23,1,1,0.0435,1.0,0.9565
41243,00018868254_26,#6***********************************************,#6**********************,23,1,1,0.0435,1.0,0.9565
18973,00025756694_20,#7***************************************11111111,#7**********************,23,1,1,0.0435,1.0,0.9565



[experian]  missing_char='-'  window=24m  24,827/100,000 rows change (24.8%)


,ZEST_KEY,zest_payment_pattern,trimmed_24,eff_old,eff_new,n_DQ30+,pct_DQ30+_old,pct_DQ30+_new,diff
70245,c36cd76c56390b4dc13eea5fce28aabf-c3116779db6f287baa7b3a7522d2f12c,#G,#G,23,1,1,0.0435,1.0,0.9565
78275,b625fea92c380555293ae03b0290b60f-5bd2a7563b6cf728f47e3d20a80a1bdb,#G,#G,23,1,1,0.0435,1.0,0.9565
93505,24aafc7407e999b67693fa4857c78513-0c631d9a95141608168cdea4e7258546,#G,#G,23,1,1,0.0435,1.0,0.9565
13260,52d55698030d26cee43c161397fb6b2d-c83ebb908fd96477da7bc608766aa597,#G,#G,23,1,1,0.0435,1.0,0.9565
44009,9c0a5bb7c4cdff959b4205c88ff88139-b0cce9d079f5322ecb1f7873809effc8,#G,#G,23,1,1,0.0435,1.0,0.9565
35835,60a4f3f036d20e55f352d10112310cb0-2bbcdad1e3af042e27f14b7ba3887220,#G,#G,23,1,1,0.0435,1.0,0.9565
93449,5d79c9082e2eeb8c997bc46e17779b33-57476a7f59bdaf68220585220300cfa7,#G,#G,23,1,1,0.0435,1.0,0.9565
77466,387480dfba8672ec5f02742d448a6822-4babccd2fd5192807b57e9779983fe74,#G,#G,23,1,1,0.0435,1.0,0.9565



[transunion]  missing_char='X'  window=24m  21,917/100,000 rows change (21.9%)


,ZEST_KEY,zest_payment_pattern,trimmed_24,eff_old,eff_new,n_DQ30+,pct_DQ30+_old,pct_DQ30+_new,diff
98020,17954364_11091764501,#G,#G,23,1,1,0.0435,1.0,0.9565
2236,17815873_10977502485,#G,#G,23,1,1,0.0435,1.0,0.9565
79426,3777634_11062169132,#G,#G,23,1,1,0.0435,1.0,0.9565
74866,17160375_10996172500,#G,#G,23,1,1,0.0435,1.0,0.9565
79416,3777634_11062169132,#G,#G,23,1,1,0.0435,1.0,0.9565
30156,17325469_10994580742,#G,#G,23,1,1,0.0435,1.0,0.9565
19847,3044259_11035777668,#L,#L,23,1,1,0.0435,1.0,0.9565
8720,13258216_11048629302,#L,#L,23,1,1,0.0435,1.0,0.9565
